# 03 — Round-by-Round Draft Contingency Sheet

This notebook consumes the VORP-ranked draft board from **Notebook 02**
and builds a personalized, round-by-round cheat sheet for a 14-team
serpentine snake draft.

**Workflow**
1. Environment & board ingestion
2. Serpentine draft-matrix generator
3. Round target & contingency engine
4. Output & export

## Cell 1 — Environment & Board Ingestion

Load the processed draft board produced by **Notebook 02** and
verify that all required columns are present before proceeding.

In [1]:
import os
import pandas as pd

# -- Paths -----------------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR     = os.path.join(PROJECT_ROOT, "data")
BOARD_CSV    = os.path.join(DATA_DIR, "draft_board_latest.csv")

# -- Load & verify ---------------------------------------------------
board = pd.read_csv(BOARD_CSV)

required_cols = [
    "vorp_rank", "pos_label", "player_name", "position_proj",
    "proj_points", "vorp", "search_rank", "adp_delta", "signal",
]
missing = [c for c in required_cols if c not in board.columns]
assert not missing, f"Missing columns: {missing}"

print(f"Loaded draft board: {len(board)} players, {len(board.columns)} columns")
print(f"\nPosition distribution:")
print(board["position_proj"].value_counts().sort_index().to_string())
print(f"\nTop 10 by VORP rank:")
display(board[["vorp_rank","pos_label","player_name","position_proj",
               "proj_points","vorp","search_rank","adp_delta","signal"]]
        .head(10))

Loaded draft board: 531 players, 16 columns

Position distribution:
position_proj
QB     84
RB    139
TE     91
WR    217

Top 10 by VORP rank:


,vorp_rank,pos_label,player_name,position_proj,proj_points,vorp,search_rank,adp_delta,signal
0,1,WR1,Ja'Marr Chase,WR,310.0,165.0,4.0,3.0,⚖️ Fair Value
1,1,WR1,Puka Nacua,WR,310.0,165.0,5.0,4.0,⚖️ Fair Value
2,3,QB1,Josh Allen,QB,360.0,160.0,4.0,1.0,⚖️ Fair Value
3,4,RB1,Bijan Robinson,RB,290.0,145.0,1.0,-3.0,⚖️ Fair Value
4,4,RB1,Jahmyr Gibbs,RB,290.0,145.0,1.0,-3.0,⚖️ Fair Value
5,4,RB1,Christian McCaffrey,RB,290.0,145.0,5.0,1.0,⚖️ Fair Value
6,4,RB1,James Cook,RB,290.0,145.0,5.0,1.0,⚖️ Fair Value
7,4,RB1,Jonathan Taylor,RB,290.0,145.0,4.0,0.0,⚖️ Fair Value
8,9,WR3,Jaxon Smith-Njigba,WR,285.0,140.0,6.0,-3.0,⚖️ Fair Value
9,9,WR3,CeeDee Lamb,WR,285.0,140.0,10.0,1.0,⚖️ Fair Value


## Cell 2 — Serpentine Draft Matrix Generator

In a 14-team serpentine draft the pick order reverses each round:
Round 1 goes 1→14, Round 2 goes 28→15, Round 3 goes 29→42, etc.

The helper below returns a list of `(round, pick)` tuples for any
draft slot.

In [2]:
def get_snake_picks(draft_slot: int, num_teams: int = 14, rounds: int = 15) -> list[tuple[int, int]]:
    """Return (round, pick_number) tuples for a serpentine draft.

    Parameters
    ----------
    draft_slot : int   Your draft position (1-indexed).
    num_teams  : int   Number of teams in the league (default 14).
    rounds     : int   Number of draft rounds to simulate (default 15).

    Returns
    -------
    list of (round_num, overall_pick) tuples.
    """
    picks = []
    for r in range(1, rounds + 1):
        if r % 2 == 1:                           # odd round  → forward
            pick = (r - 1) * num_teams + draft_slot
        else:                                     # even round → reverse
            pick = r * num_teams - draft_slot + 1
        picks.append((r, pick))
    return picks


# -- Quick demo ------------------------------------------------------
print("Example: Serpentine picks for slot 14 in a 14-team league\n")
demo = get_snake_picks(14)
for rnd, pk in demo:
    print(f"  Round {rnd:2d}  →  Pick {pk:3d}")

Example: Serpentine picks for slot 14 in a 14-team league

  Round  1  →  Pick  14
  Round  2  →  Pick  15
  Round  3  →  Pick  42
  Round  4  →  Pick  43
  Round  5  →  Pick  70
  Round  6  →  Pick  71
  Round  7  →  Pick  98
  Round  8  →  Pick  99
  Round  9  →  Pick 126
  Round 10  →  Pick 127
  Round 11  →  Pick 154
  Round 12  →  Pick 155
  Round 13  →  Pick 182
  Round 14  →  Pick 183
  Round 15  →  Pick 210


## Cell 3 — Round Target & Contingency Engine

For each of your draft picks this engine identifies:

| Tier | Meaning |
|------|---------|
| **Primary** | Best VORP in the ADP window — draft targets |
| **Secondary** | Next-best VORP pivots if primaries are taken |
| **Value** | Players with high `adp_delta` (≥ 10) — arbitrage steals |

The ADP window for each pick is `[pick - reach_buffer, pick + fall_buffer]`.
A wider `fall_buffer` catches players who may slide.

In [3]:
def generate_contingency_sheet(
    draft_slot: int,
    num_teams: int = 14,
    rounds: int = 15,
    reach_buffer: int = 4,
    fall_buffer: int = 8,
) -> pd.DataFrame:
    """Build a round-by-round contingency draft sheet.

    Parameters
    ----------
    draft_slot   : int   Your draft position (1-indexed).
    num_teams    : int   League size (default 14).
    rounds       : int   Rounds to plan (default 15).
    reach_buffer : int   How many picks *before* your pick to consider (default 4).
    fall_buffer  : int   How many picks *after* your pick to consider (default 8).

    Returns
    -------
    pd.DataFrame with one row per (round, tier) combination.
    """
    picks = get_snake_picks(draft_slot, num_teams, rounds)
    rows = []

    # Track drafted players so we can remove them across rounds
    drafted_ids = set()

    for rnd, pk in picks:
        lo = max(1, pk - reach_buffer)
        hi = pk + fall_buffer

        # Players in the ADP window who haven't been drafted yet
        pool = board[
            (~board["player_id"].isin(drafted_ids))
            & (board["search_rank"] >= lo)
            & (board["search_rank"] <= hi)
        ].copy()

        # Sort by VORP descending for the tier split
        pool = pool.sort_values("vorp", ascending=False)

        # Split into tiers
        is_value = pool["adp_delta"] >= 10
        primary   = pool[~is_value].head(3)
        secondary = pool[~is_value].iloc[3:6]
        value     = pool[is_value].head(3)

        def _rows(sub, tier):
            for _, p in sub.iterrows():
                rows.append({
                    "round":      rnd,
                    "pick":       pk,
                    "tier":       tier,
                    "vorp_rank":  int(p["vorp_rank"]),
                    "pos_label":  p["pos_label"],
                    "player":     p["player_name"],
                    "team":       p.get("team_proj", ""),
                    "position":   p["position_proj"],
                    "proj_pts":   round(p["proj_points"], 1),
                    "vorp":       round(p["vorp"], 1),
                    "search_rank": int(p["search_rank"]),
                    "adp_delta":  round(p["adp_delta"], 1),
                    "signal":     p["signal"],
                })

        _rows(primary,   "Primary")
        _rows(secondary, "Secondary")
        _rows(value,     "Value")

        # Mark primary targets as drafted for subsequent rounds
        for pid in primary["player_id"].tolist():
            drafted_ids.add(pid)

    return pd.DataFrame(rows)


# -- Quick test with slot 14 -----------------------------------------
print("=== Contingency sheet preview (Slot 14) ===\n")
test_sheet = generate_contingency_sheet(14)
print(f"Total rows: {len(test_sheet)}")
display(test_sheet.head(20))

=== Contingency sheet preview (Slot 14) ===

Total rows: 63


,round,pick,tier,vorp_rank,pos_label,player,team,position,proj_pts,vorp,search_rank,adp_delta,signal
0,1,14,Primary,9,WR3,CeeDee Lamb,DAL,WR,285.0,140.0,10,1.0,⚖️ Fair Value
1,1,14,Primary,15,WR6,Drake London,ATL,WR,250.0,105.0,18,3.0,⚖️ Fair Value
2,1,14,Primary,15,WR6,Justin Jefferson,MIN,WR,250.0,105.0,11,-4.0,⚖️ Fair Value
3,1,14,Secondary,15,WR6,A.J. Brown,NE,WR,250.0,105.0,17,2.0,⚖️ Fair Value
4,1,14,Secondary,18,RB8,Jeremiyah Love,ARI,RB,235.0,90.0,15,-3.0,⚖️ Fair Value
5,1,14,Secondary,18,RB8,Saquon Barkley,PHI,RB,235.0,90.0,12,-6.0,⚠️ Overpriced
6,2,15,Primary,15,WR6,A.J. Brown,NE,WR,250.0,105.0,17,2.0,⚖️ Fair Value
7,2,15,Primary,18,RB8,Jeremiyah Love,ARI,RB,235.0,90.0,15,-3.0,⚖️ Fair Value
8,2,15,Primary,18,RB8,Saquon Barkley,PHI,RB,235.0,90.0,12,-6.0,⚠️ Overpriced
9,2,15,Secondary,18,RB8,Ashton Jeanty,LV,RB,235.0,90.0,12,-6.0,⚠️ Overpriced


## Cell 4 — Output & Export

Run the contingency engine for two draft slots and persist the
results as both a printable text file and a structured CSV.

In [4]:
# =====================================================================
# Helper: pretty-print a contingency sheet to the console
# =====================================================================
def print_sheet(slot: int, sheet: pd.DataFrame) -> None:
    """Render a formatted round-by-round summary for the given slot."""
    picks = get_snake_picks(slot)

    print("=" * 72)
    print(f"  DRAFT CONTINGENCY SHEET  —  Slot {slot} of 14")
    print("=" * 72)

    for rnd, pk in picks:
        subset = sheet[sheet["pick"] == pk]
        if subset.empty:
            print(f"\n--- Round {rnd:2d}  |  Pick {pk:3d}  ---")
            print("  (no targets in ADP window)")
            continue

        print(f"\n--- Round {rnd:2d}  |  Pick {pk:3d}  ---")
        for tier in ["Primary", "Secondary", "Value"]:
            tier_df = subset[subset["tier"] == tier]
            if tier_df.empty:
                continue
            print(f"  [{tier}]")
            for _, r in tier_df.iterrows():
                adp_str = f"{r['adp_delta']:+.0f}"
                print(
                    f"    {r['pos_label']:5s}  "
                    f"{r['player']:<24s}  "
                    f"VORP {r['vorp']:6.1f}  "
                    f"Proj {r['proj_pts']:6.1f}  "
                    f"ADP {int(r['search_rank']):3d} ({adp_str:>5s})  "
                    f"{r['signal']}"
                )

    print("\n" + "=" * 72)


# =====================================================================
# Helper: export sheet to a text file
# =====================================================================
def export_txt(slot: int, sheet: pd.DataFrame) -> str:
    """Write a printable text file and return its path."""
    path = os.path.join(DATA_DIR, f"contingency_sheet_slot_{slot}.txt")
    picks = get_snake_picks(slot)
    lines = []
    lines.append("=" * 72)
    lines.append(f"  DRAFT CONTINGENCY SHEET  —  Slot {slot} of 14")
    lines.append("=" * 72)

    for rnd, pk in picks:
        subset = sheet[sheet["pick"] == pk]
        lines.append(f"\n--- Round {rnd:2d}  |  Pick {pk:3d}  ---")
        if subset.empty:
            lines.append("  (no targets in ADP window)")
            continue
        for tier in ["Primary", "Secondary", "Value"]:
            tier_df = subset[subset["tier"] == tier]
            if tier_df.empty:
                continue
            lines.append(f"  [{tier}]")
            for _, r in tier_df.iterrows():
                adp_str = f"{r['adp_delta']:+.0f}"
                lines.append(
                    f"    {r['pos_label']:5s}  "
                    f"{r['player']:<24s}  "
                    f"VORP {r['vorp']:6.1f}  "
                    f"Proj {r['proj_pts']:6.1f}  "
                    f"ADP {int(r['search_rank']):3d} ({adp_str:>5s})  "
                    f"{r['signal']}"
                )

    lines.append("\n" + "=" * 72)
    with open(path, "w") as f:
        f.write("\n".join(lines))
    return path


# =====================================================================
# Run for Slot 14 (the turn)
# =====================================================================
print("\n>>> Generating sheet for Slot 14 ...\n")
sheet_14 = generate_contingency_sheet(14)
print_sheet(14, sheet_14)

csv_path_14 = os.path.join(DATA_DIR, "contingency_sheet_slot_14.csv")
sheet_14.to_csv(csv_path_14, index=False)
txt_path_14 = export_txt(14, sheet_14)
print(f"\nExported: {csv_path_14}")
print(f"Exported: {txt_path_14}")


# =====================================================================
# Run for Slot 1 (the turn)
# =====================================================================
print("\n\n>>> Generating sheet for Slot 1 ...\n")
sheet_1 = generate_contingency_sheet(1)
print_sheet(1, sheet_1)

csv_path_1 = os.path.join(DATA_DIR, "contingency_sheet_slot_1.csv")
sheet_1.to_csv(csv_path_1, index=False)
txt_path_1 = export_txt(1, sheet_1)
print(f"\nExported: {csv_path_1}")
print(f"Exported: {txt_path_1}")


>>> Generating sheet for Slot 14 ...

  DRAFT CONTINGENCY SHEET  —  Slot 14 of 14

--- Round  1  |  Pick  14  ---
  [Primary]
    WR3    CeeDee Lamb               VORP  140.0  Proj  285.0  ADP  10 (   +1)  ⚖️ Fair Value
    WR6    Drake London              VORP  105.0  Proj  250.0  ADP  18 (   +3)  ⚖️ Fair Value
    WR6    Justin Jefferson          VORP  105.0  Proj  250.0  ADP  11 (   -4)  ⚖️ Fair Value
  [Secondary]
    WR6    A.J. Brown                VORP  105.0  Proj  250.0  ADP  17 (   +2)  ⚖️ Fair Value
    RB8    Jeremiyah Love            VORP   90.0  Proj  235.0  ADP  15 (   -3)  ⚖️ Fair Value
    RB8    Saquon Barkley            VORP   90.0  Proj  235.0  ADP  12 (   -6)  ⚠️ Overpriced

--- Round  2  |  Pick  15  ---
  [Primary]
    WR6    A.J. Brown                VORP  105.0  Proj  250.0  ADP  17 (   +2)  ⚖️ Fair Value
    RB8    Jeremiyah Love            VORP   90.0  Proj  235.0  ADP  15 (   -3)  ⚖️ Fair Value
    RB8    Saquon Barkley            VORP   90.0  Proj  235.0  